> **Versão pública.** Este notebook rodou no Google Colab, com os dados e artefatos numa pasta do Google Drive do grupo. Para reproduzir, veja [Como reproduzir](../../README.md#como-reproduzir).

### Fixar o Diretório

In [ ]:
wd = "/content/drive/My Drive/<PASTA_DO_PROJETO>/Base/Kaggle"

### Importar as Bibliotecas

In [ ]:
from pathlib import Path
from PIL import Image

import matplotlib.pyplot as plt

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

### Carregar os Dados

In [ ]:
ds_folder = tfds.ImageFolder(wd)
ds_train = ds_folder.as_dataset(split="train")
ds_val = ds_folder.as_dataset(split="val")
ds_test = ds_folder.as_dataset(split="test")

### Análise

#### Sumário

**Qual é o problema e quais tipos de dados estão disponíveis?**  
É um problema de classificação de imagens de doenças oculares, e estão disponíveis imagens de tomografia de coerência óptica.


**Quais as características disponíveis e como elas estão distribuídas?**  
Temos imagens de tomografia de coerência óptica em escala de cinza, com diferentes tamanhos e rotações.


**Que tipo de problema iremos lidar? Regressão, classificação, etc.?**  
Classificação.


**Qual a relação das características entre si e com a variável alvo? Elas estão correlacionadas?**  
As características são imagens, e estão relacionadas entre si, pois de certa forma representam sempre a mesma imagem base com diferentes variações por cima. Outro tipo de relacionamento entre as características que verificamos é que podem ser imagens de uma mesma pessoa. Tambem temos o problema de bordas brancas, que podem ou não ser relacionadas com 


**É um problema desbalanceado?**  
Há desbalanceamento no conjunto de treino, mas o teste é balanceado.


**Os dados precisarão de algum tipo de pré-processamento?**  
Sim, os dados precisam ser normalizados e redimensionados no mínimo. Além disso é talvez seja necessário realizar o tratamento de bordas e a aumentação de dados.


**Já há uma divisão entre treino/validação/teste?**  
Existe uma divisão entre treino, validação e teste no dataset disponível no Kaggle. No entanto iremos utilizar uma divisão que seja compatível com o outro time e outra divisão que seja compatível com o artigo original, para fins de comparação.


**Quais as métricas utilizadas neste problema? Acurácia, erro, MSRE, etc.? O que elas significam?**  
A métrica utilizada neste problema será de acurácia balanceada e curva AUC.

---
---

#### Experimentos

In [ ]:
# Quantas imagens existem em cada split?
#   test: 968
#   train: 78215

for split, n in ds_folder.info.splits.items():
    print(f"{split}: {n.num_examples}")

In [ ]:
# Quantas imagens existem em cada classe?

for folder in Path(f"{wd}/OCT2017").glob("*"):
    print(f"{folder.name}")
    for subfolder in folder.glob("*"):
        n = len(list(subfolder.glob("*.jpeg")))
        print(f"\t{subfolder.name} {n}")

In [ ]:
# Em media quantas imagens tenho de cada paciente?

In [ ]:
# Há vazamento de dados para o conjunto de testes?
#   Parece que sim

train_names = Path(f"{wd}/OCT2017/train").glob("**/*.jpeg")
train_names = [name.stem for name in train_names]
train_names = set(train_names)

test_names = Path(f"{wd}/OCT2017/test").glob("**/*.jpeg")
test_names = [name.stem for name in test_names]
test_names = set(test_names)

duplicates = train_names.intersection(test_names)

print(f"número de vazamentos: {len(duplicates)}")
print(f"exemplos: {list(duplicates)[:4]}")

In [ ]:
# São vazamentos ou amostras de um mesmo paciente?
#   São amostras de um mesmo paciente.
#   Logo essa divisão é questionável.

def plot_leak(img_stem):
    _, axs = plt.subplots(1, 2, figsize=(12, 12))
    axs = axs.flatten()
    # ------------------------------------------------------
    img_folder = img_stem.split("-")[0]
    # ------------------------------------------------------
    train_path = Path(f"{wd}/OCT2017/train/{img_folder}/{img_stem}.jpeg")
    train_img = Image.open(train_path)
    train_img = np.array(train_img)
    axs[0].imshow(train_img) #, cmap='gray')
    # ------------------------------------------------------
    test_path = Path(f"{wd}/OCT2017/test/{img_folder}/{img_stem}.jpeg")
    test_img = Image.open(test_path)
    test_img = np.array(test_img)
    axs[1].imshow(test_img) #, cmap='gray')
    # ------------------------------------------------------
    plt.show()

plot_leak("DRUSEN-609387-3")
plot_leak("CNV-172472-1")
plot_leak("NORMAL-460711-1")
plot_leak("DRUSEN-8708331-1")
plot_leak("CNV-1305450-1")
plot_leak("NORMAL-119840-1")

In [ ]:
# Como são as imagens? Que aumentações posso utilizar?
#   Preto em branco com diferentes dimensões e ângulos
#   Talvez seja o caso de alinhar as imagens com
#   algum método que avalie o gradiente das imagens.

tfds.show_examples(ds_train, ds_folder.info)